# Rizzler V11 — Unified Native Kaggle Runtime

Run this notebook on an NVIDIA accelerator with Internet enabled. This notebook launches the **same native `rizzler` package** tested in CI, plus the separate Work Agent worker. It does **not** run the older V9 server/UI. Kaggle sessions are ephemeral and are **not production persistence**. A trusted HTTPS/WSS tunnel is required for phone microphones; never expose localhost over plain HTTP.

Kaggle Secrets required: `GEMINI_API_KEY` and `RIZZLER_ACCESS_KEY` (at least 16 characters). Optional: `RIZZLER_WORK_SIGNING_KEY` and `RIZZLER_MEMORY_KEY`; encrypted memory remains disabled unless explicitly opted in. Never print the secrets.

In [ ]:
# 1 — Secrets and immutable run configuration
import os, re, subprocess, sys, time, secrets, json, urllib.request
from pathlib import Path
from kaggle_secrets import UserSecretsClient
SECRET_NAMES = ("GEMINI_API_KEY","RIZZLER_ACCESS_KEY",
                "RIZZLER_WORK_SIGNING_KEY","RIZZLER_MEMORY_KEY")
client = UserSecretsClient()
for name in SECRET_NAMES:
    if not os.environ.get(name):
        try:
            value = client.get_secret(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
if not os.environ.get("GEMINI_API_KEY") or len(os.environ.get("RIZZLER_ACCESS_KEY","")) < 16:
    raise RuntimeError("Kaggle Secrets GEMINI_API_KEY and RIZZLER_ACCESS_KEY are required")
REF=os.environ.get("RIZZLER_GIT_REF","main")
if not re.fullmatch(r"[A-Za-z0-9._/-]{1,80}",REF) or REF.startswith("-"):
    raise ValueError("Invalid Git ref")
ROOT=Path("/kaggle/working/rizzler_v11_source")
if ROOT.exists():
    raise RuntimeError("Source directory already exists; restart this session for a clean install")
subprocess.run(["git","clone","--filter=blob:none","--depth","1",
                "https://github.com/SAUNAK359/Voice-AI.git",str(ROOT)],check=True,timeout=150)
if REF!="main":
    subprocess.run(["git","-C",str(ROOT),"fetch","--depth","1","origin",REF],
                   check=True,timeout=120)
    subprocess.run(["git","-C",str(ROOT),"checkout","--detach","FETCH_HEAD"],
                   check=True,timeout=40)
REV=subprocess.check_output(["git","-C",str(ROOT),"rev-parse","HEAD"],text=True).strip()
print("Native source revision:",REV)


In [ ]:
# 2 — Install native voice, work, identity, and encrypted memory dependencies
subprocess.run([sys.executable,"-m","pip","install","-q","-e",
                str(ROOT)+"[voice,work]",
                "-r",str(ROOT/"requirements-identity.txt")],
               check=True,timeout=720)
sys.path.insert(0,str(ROOT/"src"))
from rizzler.api import create_app
from rizzler.config import Settings
from rizzler.providers import WhisperASR, KittenSpeech, GeminiLLM
from rizzler.work_worker import WorkWorker
print("Unified native application and durable-work worker imported")


In [ ]:
# 3 — Configure GPU-native app and isolated prototype storage
import torch
if not torch.cuda.is_available():
    raise RuntimeError("NVIDIA GPU not available; activate a Kaggle GPU accelerator")
print("GPU:",torch.cuda.get_device_name(0))
DATA=Path("/kaggle/working/rizzler_v11_runtime")
DATA.mkdir(exist_ok=True)
os.environ["RIZZLER_DEPLOYMENT_MODE"]="demo"
os.environ["RIZZLER_ASR_DEVICE"]="cuda"
os.environ["RIZZLER_WORK_ROOT"]=str(DATA/"jobs")
os.environ.setdefault("RIZZLER_WORK_SIGNING_KEY",secrets.token_urlsafe(48))
os.environ["RIZZLER_PREWARM"]="1"
os.environ["RIZZLER_HOST"]="127.0.0.1"
os.environ["RIZZLER_PORT"]="8765"
if os.environ.get("RIZZLER_MEMORY_ENABLED","").lower() in ("1","true","yes"):
    if len(os.environ.get("RIZZLER_MEMORY_KEY",""))<32:
        raise RuntimeError("Explicit memory opt-in requires RIZZLER_MEMORY_KEY")
    if os.environ.get("RIZZLER_DEMO_MEMORY_SINGLE_USER","")!="1":
        raise RuntimeError("Demo memory requires explicit single-user trust acknowledgement")
    os.environ["RIZZLER_MEMORY_ROOT"]=str(DATA/"private_memory")
else:
    os.environ["RIZZLER_MEMORY_ENABLED"]="0"
cfg=Settings.from_env()
print("Voice model:",cfg.asr_model,"output model:",cfg.tts_model,
      "memory opt-in:",os.environ["RIZZLER_MEMORY_ENABLED"])
print("WARNING: Kaggle /working storage is ephemeral and will not guarantee job recovery across sessions")


In [ ]:
# 4 — Launch the native application and the separately supervised Work Agent
import atexit
LOG=DATA/"logs"
LOG.mkdir(exist_ok=True)
processes=[]
def launch(name,args):
    fp=(LOG/(name+".log")).open("a",encoding="utf-8")
    process=subprocess.Popen(args,cwd=ROOT,env=os.environ.copy(),
                             stdin=subprocess.DEVNULL,stdout=fp,stderr=subprocess.STDOUT)
    fp.close()
    processes.append(process)
    return process
worker=launch("work",[sys.executable,"-m","rizzler.work_worker",
                      "--root",os.environ["RIZZLER_WORK_ROOT"]])
api=launch("api",[sys.executable,"-m","rizzler"])
def stop():
    for p in reversed(processes):
        if p.poll() is None:
            p.terminate()
    for p in reversed(processes):
        if p.poll() is None:
            try:p.wait(timeout=5)
            except subprocess.TimeoutExpired:p.kill()
atexit.register(stop)
for attempt in range(45):
    if api.poll() is not None or worker.poll() is not None:
        raise RuntimeError("Native runtime failed at startup; inspect logs in "+str(LOG))
    try:
        status=json.load(urllib.request.urlopen("http://127.0.0.1:8765/ready",timeout=2))
        if status.get("ready"):
            break
    except Exception:
        time.sleep(2)
else:
    stop()
    raise RuntimeError("Native /ready did not become healthy; examine redacted logs")
print("Native readiness passed:",status)
print("Open http://127.0.0.1:8765 only from localhost.")
print("For phone use, expose only via a trusted HTTPS/WSS reverse proxy with authentication.")


## Runtime qualification (required before calling this production-ready)

The cell above verifies native process startup and `/ready`, **not** real Whisper/Gemini/KittenTTS quality. From an authenticated browser, test voice-to-tool-to-artifact, Work Agent progress, interruption and recovery. Enable a secure proxy that supports WebSocket and never share the access key. When benchmarking, record physical first audible latency separately from server first PCM. For a persistent production host, use the documented Docker Compose deployment with real OIDC, backed-up PostgreSQL/object storage, and real GPU qualification.